# PocketAcc – fine-tune Qwen2-VL-2B on Kaggle (free GPU)

**Settings (right panel):** Accelerator **GPU T4 x2** (both GPUs are used: training and evaluation run side by side) · Internet **On**.

**How to run**
1. **Quick check (~40 min):** run the cells one by one down to *Quick check*. If a step fails, the notebook stops with an error – send that error back.
2. **Full run (~3–5 h):** *Save Version → Save & Run All (Commit)*. It runs in the background, so you can close the laptop. When it finishes, open that version's **Output** tab and download `pocketacc_results.zip`.

Every step stops the notebook if it fails, so a broken step cannot silently waste hours.

In [ ]:
!nvidia-smi

In [ ]:
REPO = "https://github.com/anshace07/PocketAcc.git"
!git clone -q $REPO /kaggle/working/pocketacc
%cd /kaggle/working/pocketacc
!git log --oneline -3
!apt-get -qq install -y poppler-utils tesseract-ocr > /dev/null
!pip -q install -r requirements.txt -r requirements-gpu.txt
# the Nov 2026 fix must be in the cloned code (vision encoder kept out of 4-bit) - see docs/FAILURES.md
assert "llm_int8_skip_modules" in open("src/vlm.py").read(), "OLD CODE: push the 4-bit vision fix to GitHub first"

In [ ]:
# helpers: run a step, show its output, and STOP the notebook if it fails
import os, subprocess, time
from pathlib import Path
N_GPU = len(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines())
Path("logs").mkdir(exist_ok=True)
print("GPUs:", N_GPU)

def sh(cmd, gpu=0):
    """Run one command on one GPU with live output; raise if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

def parallel(jobs):
    """Run jobs side by side, one per GPU (T4 x2 = about 2x faster); with one GPU, one after another.
    Prints progress every 2 minutes; full logs in logs/<name>.log."""
    if N_GPU < 2:
        for cmd in jobs.values():
            sh(cmd)
        return
    procs = {}
    for i, (name, cmd) in enumerate(jobs.items()):
        procs[name] = subprocess.Popen(cmd, shell=True, stdout=open(f"logs/{name}.log", "w"), stderr=subprocess.STDOUT,
                                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(i % N_GPU)))
        print(f"started {name} on GPU {i % N_GPU}   (log: logs/{name}.log)", flush=True)
    while any(p.poll() is None for p in procs.values()):
        time.sleep(120)
        for name, p in procs.items():
            lines = [l for l in open(f"logs/{name}.log", errors="ignore").read().splitlines() if l.strip()]
            state = "running" if p.poll() is None else f"finished (exit {p.returncode})"
            print(f"[{time.strftime('%H:%M')}] {name}: {state} | {lines[-1][:140] if lines else ''}", flush=True)
    failed = [n for n, p in procs.items() if p.returncode]
    for n in failed:
        print(f"\n----- last lines of logs/{n}.log -----")
        print("\n".join(open(f"logs/{n}.log", errors="ignore").read().splitlines()[-30:]))
    if failed:
        raise RuntimeError(f"FAILED: {failed}")

## Step 1–3: build the page-level dataset (≈20 min on CPU, deterministic seeds)

In [ ]:
sh("python src/build_dataset.py --out data/vlm")

In [ ]:
sh("python scripts/smoke_test.py data/vlm")   # tiny-model test of the training code (1–2 min)

## Step 4a: will it fit? (peak GPU memory for inference and one training step)

In [ ]:
sh("python scripts/vram_test.py --data data/vlm")

## Quick check (~5 min): 2 training steps, then read 1 page with the result

Catches crashes before the long run. The output is cut at 300 tokens, so `rows` may be 0 here.
What matters: the `raw:` text should look like statement text / JSON, **not** empty or `!!!!!`.

In [ ]:
sh("python src/finetune.py --data data/vlm --out runs/quick --limit 8 --max-steps 2 --grad-accum 1 "
   "--log-every 1 --eval-every 2 --val-samples 2")
sh("python src/evaluate.py predict --system vlm --no-4bit --adapter runs/quick/best_adapter --split test_unseen "
   "--limit 1 --max-new-tokens 300 --out runs/quick/pred.jsonl")
sh("rm -rf runs/quick")
print("\nQUICK CHECK PASSED - now: Save Version -> Save & Run All")

## Step 4b + 4c: QLoRA fine-tuning (GPU 0) while the zero-shot baseline runs (GPU 1)

Tesseract results for these pages are already in the repo, so its two lines finish instantly.

In [ ]:
sh("python src/evaluate.py predict --system tesseract --split test_unseen --limit 24 --out results/tess_unseen.jsonl")
sh("python src/evaluate.py predict --system tesseract --split test_seen   --limit 24 --out results/tess_seen.jsonl")
parallel({
    # the run that produced adapters/lora_r8 (Nov 2026): 300 training pages, pages <= 900k pixels;
    # expandable_segments fixed an out-of-memory error in the first attempt; if it still runs out of
    # memory, the second command retries with smaller pages instead of failing the whole run
    "finetune":  "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/finetune.py --data data/vlm --out runs/lora_r8 --rank 8 --epochs 1 --grad-accum 8 --limit 300 --max-pixels 900000"
                 " || PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/finetune.py --data data/vlm --out runs/lora_r8 --rank 8 --epochs 1 --grad-accum 8 --limit 300 --max-pixels 600000",
    "zero_shot": "python src/evaluate.py predict --system vlm --no-4bit --split test_unseen --limit 24 --max-pixels 900000 --out results/base_unseen.jsonl"
                 " && python src/evaluate.py predict --system vlm --no-4bit --split test_seen --limit 24 --max-pixels 900000 --out results/base_seen.jsonl",
})

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("runs/lora_r8/training_log.csv")
ax = log.plot(x="step", y="train_loss", figsize=(7,3.5)); log.dropna(subset=["val_loss"]).plot(x="step", y="val_loss", ax=ax, marker="o")
plt.ylabel("loss"); plt.title("QLoRA fine-tuning"); plt.savefig("results/loss_curve.png", dpi=150, bbox_inches="tight"); plt.show()

## Step 4d: evaluate the fine-tuned model (unseen banks on GPU 0, seen banks on GPU 1)

In [ ]:
parallel({
    "ft_unseen": "python src/evaluate.py predict --system vlm --no-4bit --adapter runs/lora_r8/best_adapter --split test_unseen --limit 24 --max-pixels 900000 --out results/ft_unseen.jsonl",
    "ft_seen":   "python src/evaluate.py predict --system vlm --no-4bit --adapter runs/lora_r8/best_adapter --split test_seen   --limit 24 --max-pixels 900000 --out results/ft_seen.jsonl",
})

## Scores – every system on the **same 24 pages** per split (`--limit 24`)

In [ ]:
for name in ["tess", "base", "ft"]:
    for split in ["seen", "unseen"]:
        print(f"\n===== {name} / test_{split} =====")
        sh(f"python src/evaluate.py score results/{name}_{split}.jsonl --split test_{split} --limit 24 "
           f"--json-out results/score24_{name}_{split}.json")

In [ ]:
!zip -qr /kaggle/working/pocketacc_results.zip results logs runs/lora_r8/best_adapter runs/lora_r8/training_log.csv runs/lora_r8/run_config.json && ls -lh /kaggle/working/pocketacc_results.zip